# Imports

In [1]:
import copy

import numpy as np
import torch
import torch.nn.functional as F

%load_ext autoreload
%autoreload 2

In [2]:
from config import RunConfig
from sweep import build_eval_loaders
from models import load_checkpoint
from dropout import reset_dropout, configure_dropout, configure_pre_residual_dropout
from inference import build_baseline_cache, compute_psu_and_shift
from detection import (
    attack_success_rate,
    clean_accuracy,
    threshold_from_validation,
    detection_rates,
    auroc,
)

# Testing

In [25]:
!ls vit_b_16_weights

cifar100_badnet_0_01	  cifar10_blind_0_1	   gtsrb_bpp_0_05
cifar100_badnet_0_05	  cifar10_bpp_0_01	   gtsrb_bpp_0_1
cifar100_badnet_0_1	  cifar10_bpp_0_05	   gtsrb_inputaware_0_01
cifar100_blended_0_01	  cifar10_bpp_0_1	   gtsrb_inputaware_0_05
cifar100_blended_0_05	  cifar10_inputaware_0_01  gtsrb_inputaware_0_1
cifar100_blended_0_1	  cifar10_inputaware_0_05  gtsrb_lf_0_01
cifar100_blind_0_1	  cifar10_inputaware_0_1   gtsrb_lf_0_05
cifar100_bpp_0_01	  cifar10_lc_0_01	   gtsrb_lf_0_1
cifar100_bpp_0_05	  cifar10_lc_0_05	   gtsrb_lira_0_1
cifar100_bpp_0_1	  cifar10_lc_0_1	   gtsrb_ssba_0_01
cifar100_inputaware_0_01  cifar10_lf_0_01	   gtsrb_ssba_0_05
cifar100_inputaware_0_05  cifar10_lf_0_05	   gtsrb_ssba_0_1
cifar100_inputaware_0_1   cifar10_lf_0_1	   gtsrb_trojannn_0_01
cifar100_lc_0_01	  cifar10_lira_0_1	   gtsrb_trojannn_0_05
cifar100_lf_0_01	  cifar10_sig_0_01	   gtsrb_trojannn_0_1
cifar100_lf_0_05	  cifar10_sig_0_05	   gtsrb_wanet_0_01
cifar100_lf_0_1		  cifar10_sig_0_1	   gtsrb_wa

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def run_all(
    folder="cifar10_badnet_0_1",
    placement="pre_residual",
    dropout_rate=0.1,
    quantile=0.25,
):
    config = RunConfig(
        architecture="vit",
        weights_dir="vit_b_16_weights",
        dropout_placement=placement,
    )
    # folder = "cifar10_badnet_0_1"
    checkpoint = f"{config.weights_dir}/{folder}/attack_result.pt"
    val_loader, clean_loader, backdoor_loader = build_eval_loaders(config, folder)
    model = load_checkpoint(config.architecture, checkpoint, device)
    reset_dropout(model, config.dropout_placement)
    print("ASR", attack_success_rate(model, backdoor_loader, device, True))
    print("CA", clean_accuracy(model, clean_loader, device, True))

    count = configure_pre_residual_dropout(model, 0.0)
    print("dropout modules touched:", count)

    reset_dropout(model, config.dropout_placement)
    val_cache = build_baseline_cache(model, val_loader, device, True)
    clean_cache = build_baseline_cache(model, clean_loader, device, True)
    backdoor_cache = build_baseline_cache(model, backdoor_loader, device, True)

    configure_dropout(model, dropout_rate, config.dropout_placement)
    val_scores, val_shift = compute_psu_and_shift(
        model, val_loader, val_cache, device, 3, True, 0
    )
    clean_scores, clean_shift = compute_psu_and_shift(
        model, clean_loader, clean_cache, device, 3, True, 0
    )
    backdoor_scores, backdoor_shift = compute_psu_and_shift(
        model, backdoor_loader, backdoor_cache, device, 3, True, 0
    )

    print(
        "mean PSU clean",
        clean_scores.mean().item(),
        "backdoor",
        backdoor_scores.mean().item(),
    )
    print("shift clean", clean_shift, "backdoor", backdoor_shift)

    threshold = threshold_from_validation(val_scores, quantile=quantile)
    tpr, fpr = detection_rates(clean_scores, backdoor_scores, threshold)
    print(
        "threshold",
        threshold,
        "TPR",
        tpr,
        "FPR",
        fpr,
        "AUROC",
        auroc(clean_scores, backdoor_scores),
    )

In [45]:
config = RunConfig(
    architecture="vit",
    weights_dir="vit_b_16_weights",
    dropout_placement="pre_residual",
)
folder = "tiny_wanet_0_1"
checkpoint = f"{config.weights_dir}/{folder}/attack_result.pt"

In [47]:
val_loader, clean_loader, backdoor_loader = build_eval_loaders(config, folder)

/lustre/home/pstika/projects/PSBD-ViT/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
Seed set to 0


In [46]:
model = load_checkpoint(config.architecture, checkpoint, device)
reset_dropout(model, config.dropout_placement)

In [48]:
print("ASR", attack_success_rate(model, backdoor_loader, device, True))
print("CA", clean_accuracy(model, clean_loader, device, True))

ASR 0.9959748427672956
CA 0.6316981132075472


In [49]:
count = configure_pre_residual_dropout(model, 0.0)
print("dropout modules touched:", count)

dropout modules touched: 37


In [50]:
reset_dropout(model, "pre_residual")
val_cache = build_baseline_cache(model, val_loader, device, True)
clean_cache = build_baseline_cache(model, clean_loader, device, True)
backdoor_cache = build_baseline_cache(model, backdoor_loader, device, True)

In [51]:
dropout_rate = 0.1
configure_dropout(model, dropout_rate, "pre_residual")
val_scores, val_shift = compute_psu_and_shift(
    model, val_loader, val_cache, device, 3, True, 0
)
clean_scores, clean_shift = compute_psu_and_shift(
    model, clean_loader, clean_cache, device, 3, True, 0
)
backdoor_scores, backdoor_shift = compute_psu_and_shift(
    model, backdoor_loader, backdoor_cache, device, 3, True, 0
)

print(f"\n\n{'-' * 20} dropout rate {dropout_rate} {'-' * 20}")
print(
    "mean PSU clean",
    clean_scores.mean().item(),
    "backdoor",
    backdoor_scores.mean().item(),
)
print("shift clean", clean_shift, "backdoor", backdoor_shift)

Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.1 --------------------
mean PSU clean 0.7474791407585144 backdoor 0.25705406069755554
shift clean 0.9110272536687631 backdoor 0.08419287211740042


In [52]:
quantile = 0.25
threshold = threshold_from_validation(val_scores, quantile=quantile)
tpr, fpr = detection_rates(clean_scores, backdoor_scores, threshold)
print(
    "threshold",
    threshold,
    "TPR",
    tpr,
    "FPR",
    fpr,
    "AUROC",
    auroc(clean_scores, backdoor_scores),
)

threshold 0.52108234167099 TPR 0.8759748339653015 FPR 0.22905659675598145 AUROC 0.9104501878881374


In [54]:
for quantile in [0.01, 0.05, 0.1, 0.15, 0.2, 0.25]:
    threshold = threshold_from_validation(val_scores, quantile=quantile)
    tpr, fpr = detection_rates(clean_scores, backdoor_scores, threshold)
    print(
        "quantile",
        quantile,
        "threshold",
        threshold,
        "TPR",
        tpr,
        "FPR",
        fpr,
        "AUROC",
        auroc(clean_scores, backdoor_scores),
    )

quantile 0.01 threshold 0.1452169418334961 TPR 0.37987419962882996 FPR 0.014213836751878262 AUROC 0.9104501878881374
quantile 0.05 threshold 0.23498213291168213 TPR 0.5008804798126221 FPR 0.052452828735113144 AUROC 0.9104501878881374
quantile 0.1 threshold 0.3188096284866333 TPR 0.6124528050422668 FPR 0.09496854990720749 AUROC 0.9104501878881374
quantile 0.15 threshold 0.3862903416156769 TPR 0.751069188117981 FPR 0.13547170162200928 AUROC 0.9104501878881374
quantile 0.2 threshold 0.45358380675315857 TPR 0.8236477971076965 FPR 0.17899371683597565 AUROC 0.9104501878881374
quantile 0.25 threshold 0.52108234167099 TPR 0.8759748339653015 FPR 0.22905659675598145 AUROC 0.9104501878881374


In [38]:
for dropout_rate in [0.05, 0.1, 0.2, 0.3, 0.4]:
    configure_dropout(model, dropout_rate, "pre_residual")
    val_scores, val_shift = compute_psu_and_shift(
        model, val_loader, val_cache, device, 3, True, 0
    )
    clean_scores, clean_shift = compute_psu_and_shift(
        model, clean_loader, clean_cache, device, 3, True, 0
    )
    backdoor_scores, backdoor_shift = compute_psu_and_shift(
        model, backdoor_loader, backdoor_cache, device, 3, True, 0
    )

    print(f"\n\n{'-' * 20} dropout rate {dropout_rate} {'-' * 20}")
    print(
        "mean PSU clean",
        clean_scores.mean().item(),
        "backdoor",
        backdoor_scores.mean().item(),
    )
    print("shift clean", clean_shift, "backdoor", backdoor_shift)

    threshold = threshold_from_validation(val_scores, 0.25)
    tpr, fpr = detection_rates(clean_scores, backdoor_scores, threshold)
    print(
        "dropout rate",
        dropout_rate,
        "threshold",
        threshold,
        "TPR",
        tpr,
        "FPR",
        fpr,
        "AUROC",
        auroc(clean_scores, backdoor_scores),
    )

Seed set to 0
Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.1 --------------------
mean PSU clean 0.7474791407585144 backdoor 0.25705406069755554
shift clean 0.9110272536687631 backdoor 0.08419287211740042
dropout rate 0.1 threshold 0.52108234167099 TPR 0.8759748339653015 FPR 0.22905659675598145 AUROC 0.9104501878881374


Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.2 --------------------
mean PSU clean 0.773561418056488 backdoor 0.6515788435935974
shift clean 0.9916142557651991 backdoor 0.18574423480083857
dropout rate 0.2 threshold 0.5361865758895874 TPR 0.25182390213012695 FPR 0.22767294943332672 AUROC 0.6955019421700092


Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.3 --------------------
mean PSU clean 0.7749451398849487 backdoor 0.7648652195930481
shift clean 0.9940461215932914 backdoor 0.3538784067085954
dropout rate 0.3 threshold 0.5361977815628052 TPR 0.10238993912935257 FPR 0.226289302110672 AUROC 0.6069390372216288


Seed set to 0
Seed set to 0




-------------------- dropout rate 0.4 --------------------
mean PSU clean 0.7753081917762756 backdoor 0.8232957720756531
shift clean 0.9945911949685534 backdoor 0.5238155136268344
dropout rate 0.4 threshold 0.5347135066986084 TPR 0.052075471729040146 FPR 0.22540880739688873 AUROC 0.5425274000237332


In [39]:
for dropout_rate in [0.05, 0.1, 0.2, 0.3, 0.4]:
    configure_dropout(model, dropout_rate, "post_residual")
    val_scores, val_shift = compute_psu_and_shift(
        model, val_loader, val_cache, device, 3, True, 0
    )
    clean_scores, clean_shift = compute_psu_and_shift(
        model, clean_loader, clean_cache, device, 3, True, 0
    )
    backdoor_scores, backdoor_shift = compute_psu_and_shift(
        model, backdoor_loader, backdoor_cache, device, 3, True, 0
    )

    print(f"\n\n{'-' * 20} dropout rate {dropout_rate} {'-' * 20}")
    print(
        "mean PSU clean",
        clean_scores.mean().item(),
        "backdoor",
        backdoor_scores.mean().item(),
    )
    print("shift clean", clean_shift, "backdoor", backdoor_shift)

    threshold = threshold_from_validation(val_scores, 0.25)
    tpr, fpr = detection_rates(clean_scores, backdoor_scores, threshold)
    print(
        "dropout rate",
        dropout_rate,
        "threshold",
        threshold,
        "TPR",
        tpr,
        "FPR",
        fpr,
        "AUROC",
        auroc(clean_scores, backdoor_scores),
    )

Seed set to 0
Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.05 --------------------
mean PSU clean 0.7754592895507812 backdoor 0.9107134938240051
shift clean 0.9947169811320755 backdoor 0.6351781970649896
dropout rate 0.05 threshold 0.5379939675331116 TPR 0.004402515944093466 FPR 0.22679245471954346 AUROC 0.44647180886832005


Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.1 --------------------
mean PSU clean 0.7755475044250488 backdoor 0.9602721929550171
shift clean 0.9951781970649896 backdoor 0.7715303983228512
dropout rate 0.1 threshold 0.5375857353210449 TPR 0.002767295576632023 FPR 0.2266666740179062 AUROC 0.34977542027609665


Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.2 --------------------
mean PSU clean 0.7755662798881531 backdoor 0.9809811115264893
shift clean 0.9948427672955975 backdoor 0.8703144654088051
dropout rate 0.2 threshold 0.5396169424057007 TPR 0.002389937173575163 FPR 0.22817610204219818 AUROC 0.265293991535145


Seed set to 0
Seed set to 0
Seed set to 0




-------------------- dropout rate 0.3 --------------------
mean PSU clean 0.7755704522132874 backdoor 0.9781697988510132
shift clean 0.9953459119496856 backdoor 0.8215094339622642
dropout rate 0.3 threshold 0.5391876697540283 TPR 0.002515723230317235 FPR 0.22867923974990845 AUROC 0.2785493295360152


Seed set to 0
Seed set to 0




-------------------- dropout rate 0.4 --------------------
mean PSU clean 0.7756269574165344 backdoor 0.9718363881111145
shift clean 0.9952201257861635 backdoor 0.7969811320754717
dropout rate 0.4 threshold 0.5373314023017883 TPR 0.0026415095198899508 FPR 0.22679245471954346 AUROC 0.2985008820853605


# Setting up model dropout

In [80]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Dropout):
        print(name)

1.encoder.dropout
1.encoder.layers.encoder_layer_0.dropout
1.encoder.layers.encoder_layer_0.mlp.2
1.encoder.layers.encoder_layer_0.mlp.4
1.encoder.layers.encoder_layer_1.dropout
1.encoder.layers.encoder_layer_1.mlp.2
1.encoder.layers.encoder_layer_1.mlp.4
1.encoder.layers.encoder_layer_2.dropout
1.encoder.layers.encoder_layer_2.mlp.2
1.encoder.layers.encoder_layer_2.mlp.4
1.encoder.layers.encoder_layer_3.dropout
1.encoder.layers.encoder_layer_3.mlp.2
1.encoder.layers.encoder_layer_3.mlp.4
1.encoder.layers.encoder_layer_4.dropout
1.encoder.layers.encoder_layer_4.mlp.2
1.encoder.layers.encoder_layer_4.mlp.4
1.encoder.layers.encoder_layer_5.dropout
1.encoder.layers.encoder_layer_5.mlp.2
1.encoder.layers.encoder_layer_5.mlp.4
1.encoder.layers.encoder_layer_6.dropout
1.encoder.layers.encoder_layer_6.mlp.2
1.encoder.layers.encoder_layer_6.mlp.4
1.encoder.layers.encoder_layer_7.dropout
1.encoder.layers.encoder_layer_7.mlp.2
1.encoder.layers.encoder_layer_7.mlp.4
1.encoder.layers.encoder_layer

In [6]:
import torch
from config import RunConfig
from checkpoint_eval import read_checkpoint_metadata, build_eval_loaders_from_checkpoint
from models import load_checkpoint

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [4]:
attack_result = torch.load(
    "checkpoints/cifar10_swin_adaptive_blend_0_005/attack_result.pt"
)

In [10]:
list(attack_result.keys())

['model', 'num_classes']

In [ ]:
# how to check what is inside attack_result?

In [8]:
folder = "cifar10_swin_adaptive_blend_0_005"
weights_dir = "checkpoints"

path = f"{weights_dir}/{folder}/attack_result.pt"
meta = read_checkpoint_metadata(path)
print("architecture:", meta.get("architecture"))
print("dataset:     ", meta.get("dataset"))
print("attack:      ", meta.get("attack"))
print("poison_rate: ", meta.get("poison_rate"))
print("clean_acc:   ", meta.get("clean_accuracy"))

KeyError: "Checkpoint checkpoints/cifar10_swin_adaptive_blend_0_005/attack_result.pt is missing metadata ['dataset', 'attack', 'target_label']. Retrain with train_backdoor.py, which records it, or provide a BackdoorBench bd_test_dataset folder."

In [11]:
import os
import glob
import torch

WEIGHTS_DIR = "checkpoints"

In [16]:
def parse_folder(folder):
    parts = folder.split("_")
    architecture = "swin" if "swin" in parts else "vit"
    parts = [p for p in parts if p not in ("vit", "swin")]
    dataset = parts[0]
    rest = parts[1:]

    if rest == ["benign"]:
        return {
            "architecture": architecture,
            "dataset": dataset,
            "attack": "benign",
            "poison_rate": 0.0,
            "target_label": 0,
        }

    poison_rate = float(f"{rest[-2]}.{rest[-1]}")  # last two tokens, e.g. 0 and 005
    attack = "_".join(rest[:-2])
    target_label = "all" if "a2a" in attack else 0
    return {
        "architecture": architecture,
        "dataset": dataset,
        "attack": attack,
        "poison_rate": poison_rate,
        "target_label": target_label,
    }

In [17]:
for f in [
    "vit_cifar100_adaptive_blend_0_005",
    "cifar10_swin_badnet_a2a_0_1",
    "cifar10_badnet_a2o_0_1",
    "vit_cifar10_benign",
]:
    print(f, "->", parse_folder(f))

vit_cifar100_adaptive_blend_0_005 -> {'architecture': 'vit', 'dataset': 'cifar100', 'attack': 'adaptive_blend', 'poison_rate': 0.005, 'target_label': 0}
cifar10_swin_badnet_a2a_0_1 -> {'architecture': 'swin', 'dataset': 'cifar10', 'attack': 'badnet_a2a', 'poison_rate': 0.1, 'target_label': 'all'}
cifar10_badnet_a2o_0_1 -> {'architecture': 'vit', 'dataset': 'cifar10', 'attack': 'badnet_a2o', 'poison_rate': 0.1, 'target_label': 0}
vit_cifar10_benign -> {'architecture': 'vit', 'dataset': 'cifar10', 'attack': 'benign', 'poison_rate': 0.0, 'target_label': 0}


In [18]:
from models import load_checkpoint


def load_own(folder, weights_dir="checkpoints"):
    info = parse_folder(folder)
    path = f"{weights_dir}/{folder}/attack_result.pt"
    model = load_checkpoint(info["architecture"], path, device)
    return model, info

In [19]:
model, info = load_own("vit_cifar10_badnet_a2o_0_1")
print(info)
print("loaded:", type(model[1]).__name__)

{'architecture': 'vit', 'dataset': 'cifar10', 'attack': 'badnet_a2o', 'poison_rate': 0.1, 'target_label': 0}
loaded: VisionTransformer


In [22]:
swin = load_checkpoint(
    "swin", "checkpoints/cifar10_swin_adaptive_blend_0_005/attack_result.pt", device
)

In [26]:
a = sum(p.numel() for p in swin.parameters())
b = sum(p.numel() for p in model.parameters())
print(f"{a:,} parameters in swin, {b:,} parameters in vit")

48,844,948 parameters in swin, 85,806,346 parameters in vit
